# Lab 01 — RAG from Scratch

**Pairs with:** [Course 06 · RAG](https://psssnikhil.github.io/ai-engineering-handbook/build/module-09-rag-retrieval-augmented-generation/)

You will build a complete Retrieval-Augmented Generation pipeline with **no frameworks** — just chunking, TF-IDF retrieval, and LLM generation (supporting **OpenAI** or **Anthropic**). By the end you'll understand exactly what LangChain and LlamaIndex abstract away, and you'll be able to implement this in a coding interview.

The pipeline:

```
documents → chunk → index (TF-IDF) → retrieve top-k → build grounded prompt → LLM (OpenAI / Claude) → cited answer
```

**Prerequisites:** `pip install -r requirements.txt` and set `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` in your environment. A full run costs a few cents.

In [ ]:
import os
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Flexible Multi-Provider LLM Client Setup (OpenAI or Anthropic)
if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    llm_client = OpenAI()
    MODEL_NAME = "gpt-4o-mini"
    PROVIDER = "openai"
    print(f"✓ Initialized OpenAI Provider with model: {MODEL_NAME}")
elif os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic
    llm_client = anthropic.Anthropic()
    MODEL_NAME = "claude-3-5-sonnet-20241022"
    PROVIDER = "anthropic"
    print(f"✓ Initialized Anthropic Provider with model: {MODEL_NAME}")
else:
    raise ValueError("Please export OPENAI_API_KEY or ANTHROPIC_API_KEY in your environment.")

def generate_llm_response(prompt: str) -> str:
    if PROVIDER == "openai":
        res = llm_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300
        )
        return res.choices[0].message.content
    else:
        res = llm_client.messages.create(
            model=MODEL_NAME,
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}]
        )
        return res.content[0].text

## Step 1: Sample Documents

Let's create a small corpus of company policy documents.

In [ ]:
DOCUMENTS = [
    """Acme Corp Remote Work Policy (2026):
    Employees may work remotely up to 3 days per week with manager approval.
    A home office stipend of $500/year is available for hardware purchase.
    Core working hours are 10 AM to 4 PM Eastern Time.""",
    
    """Acme Corp Vacation & Leave Policy:
    Full-time employees receive 20 days of paid time off (PTO) annually.
    Unused PTO rolls over up to a maximum of 5 days into the following calendar year.
    Parental leave provides 16 weeks of fully paid leave for primary caregivers.""",
    
    """Acme Corp Security & Compliance:
    All laptops must run full-disk encryption and company MDM agent.
    Passwords must be at least 16 characters and rotated every 90 days.
    Multi-factor authentication (MFA) is required for all internal tools."""
]

## Step 2: Indexing with TF-IDF Vectorizer

We convert our document corpus into TF-IDF vector embeddings for fast mathematical cosine retrieval.

In [ ]:
vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(DOCUMENTS)
print(f"Indexed {len(DOCUMENTS)} documents. Vocabulary size: {len(vectorizer.get_feature_names_out())}")

## Step 3: Retrieval Function

Given a user query, compute cosine similarity against indexed document vectors and return top-k matches.

In [ ]:
def retrieve_top_k(query: str, k: int = 1):
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, doc_vectors)[0]
    top_k_indices = np.argsort(similarities)[::-1][:k]
    
    retrieved = []
    for idx in top_k_indices:
        retrieved.append((DOCUMENTS[idx], similarities[idx]))
    return retrieved

# Test retrieval
query = "How much is the home office stipend?"
results = retrieve_top_k(query, k=1)
print(f"Top Match (Score {results[0][1]:.4f}):\n{results[0][0]}")

## Step 4: Grounded Generation

Inject the retrieved context into a grounded system prompt to generate accurate, cited answers.

In [ ]:
def answer_query(user_query: str) -> str:
    retrieved = retrieve_top_k(user_query, k=1)
    context_text = retrieved[0][0]
    
    prompt = f"""Answer the question using ONLY the provided context.
If the answer cannot be found in the context, say 'I cannot answer based on the provided context.'

Context:
{context_text}

Question: {user_query}
Answer:"""
    
    return generate_llm_response(prompt)

answer = answer_query("What is the annual home office stipend amount?")
print(f"\nQUERY RESPONSE:\n{answer}")